# Local SME Underwriting Agents Notebook

Full Credit Memo workflow: Business Activity Analysis -> Credit Relationship -> Financial Analysis -> Credit Proposal -> Risk Assessment -> Credit Memo.

In [ ]:
from __future__ import annotations

import hashlib
import httpx
import json
import os
import re
import shutil
import sys
import threading
import time
import unicodedata

from dataclasses import asdict, dataclass, field
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Any, Literal, TypedDict

from dotenv import load_dotenv


PROJECT_ROOT = Path().resolve()
load_dotenv(PROJECT_ROOT / ".env", override=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
USER_PROMPT = "Hãy phân tích hoạt động kinh doanh cho khách hàng này. Theo chương trình PLO"
TESTCASE_ID = "case_1"

# Có thể truyền file hoặc folder. Folder sẽ được scan đệ quy các file
# PDF/XLS/XLSX/CSV/TXT/MD.
INPUT_PATHS = [
    str(PROJECT_ROOT / "testing" / "samples" / TESTCASE_ID),
    # "/absolute/path/to/BCTC.pdf",
]

CONVERSATION_HISTORY = [
    # {"role": "user", "content": "..."},
    # {"role": "assistant", "content": "..."},
]

OUTPUT_DIR = PROJECT_ROOT / "logs"
MAX_CHARS_PER_DOCUMENT = 120_000

In [ ]:
# LangSmith tracing (optional).
# from src.agents.tracing import run_supervisor_with_optional_trace


In [ ]:
# LLM client factory & Config — extracted to underwriting.config.
from src.config import Config, build_llm

config = Config(
    decision_llm=build_llm(
        "MODEL_DECISION", 
        temperature=0.1
    ),
    document_llm=build_llm(
        "MODEL_DOCUMENT",
        temperature=0.5
    ),
    analysis_llm=build_llm(
        "MODEL_ANALYZER",
        temperature=0.1,
    ),
    credit_memo_llm=build_llm(
        "MODEL_CREDIT_MEMO",
        temperature=0.1,
    ),
    guardrail_llm=build_llm(
        "MODEL_GUARDRAIL",
        temperature=0.0,
    ),
    bctc_extraction_llm=build_llm(
        "MODEL_BCTC_EXTRACTION",
        temperature=0.0,
    ),
    proposal_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    cic_s10a_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    cic_r21_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    sitevisit_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)

print(
    "LLM availability:",
    {
        "decision": bool(config.decision_llm),
        "document": bool(config.document_llm),
        "analysis": bool(config.analysis_llm),
        "credit_memo": bool(config.credit_memo_llm),
        "guardrail": bool(config.guardrail_llm),
        "bctc_extraction": bool(config.bctc_extraction_llm),
        "proposal_extraction": bool(config.proposal_extraction_llm),
        "cic_s10a_extraction": bool(config.cic_s10a_extraction_llm),
        "cic_r21_extraction": bool(config.cic_r21_extraction_llm),
        "sitevisit_extraction": bool(config.sitevisit_extraction_llm),
    },
)

In [ ]:
# Shared types & helpers — see underwriting.types.
from src.types import (
    AgentName,
    WorkflowMode,
    DocumentAgentName,
    ClassifiedDocument,
    UnderwritingGraphState,
    to_dict_list,
    extract_text_from_agent_output,
    truncate_text,
)


In [ ]:
# Database tools.
# from src.agents.tools import (
#     configure_database_executor,
#     get_database_tools,
#     DATABASE_TOOLS,
#     FINANCIAL_DATABASE_TOOLS,
#     BUSINESS_ACTIVITY_DATABASE_TOOLS,
#     CREDIT_RELATIONSHIP_DATABASE_TOOLS,
#     RISK_ASSESSMENT_DATABASE_TOOLS,
# )


In [ ]:
# Document extraction & classification.
from src.agents.document_classification import (
    SUPPORTED_EXTENSIONS,
    VALID_DOCUMENT_AGENTS,
    compute_file_hash,
    resolve_input_path,
    discover_documents,
    normalize_text,
    document_type_scores,
    rule_classify_document,
)

# Document routing matrix (src/matrix/document_matrix.yaml): document type ->
# consuming agents. Loading it here surfaces a malformed matrix immediately
# instead of midway through a workflow run.
from src.matrix.document_matrix import (
    agent_relevance_for_type,
    all_types,
    load_matrix,
    primary_agent_for_type,
)

_matrix = load_matrix()
print(
    f"Document matrix v{_matrix.version}: {len(all_types())} document types, "
    f"loan programs = {', '.join(_matrix.loan_programs)}"
)


In [ ]:
# Specialist agents & memo composer — see underwriting.agents.
from src.agents.specialist import (
    SpecialistAgent,
    BusinessActivityAnalysis,
    FinancialAnalysis,
    CreditRelationshipAnalysis,
    CreditProposalAnalysis,
    RiskAssessment,
    build_credit_memo,
    CreditMemoComposerAgent,
)

from src.agents.guardrails import (
    LocalGuardrails,
    WebSearchProcessorAgent,
)

from src.agents.supervisor import Supervisor

In [ ]:
from IPython.display import Markdown, display

from src.utils.common import show_graph

supervisor = Supervisor(config)

display(Markdown("### Workflow Graph"))
display(show_graph(supervisor.workflow_graph))

workflow_runner = globals().get("run_supervisor_with_optional_trace")
if workflow_runner:
    result = workflow_runner(
        supervisor,
        USER_PROMPT,
        INPUT_PATHS,
        CONVERSATION_HISTORY,
    )
else:
    result = supervisor.process(USER_PROMPT, INPUT_PATHS, CONVERSATION_HISTORY)

display(Markdown(result["response"]))

print("\n--- Agent Name ---")
print(result["agent_name"])

print("\n--- Steps ---")
for step in result["steps"]:
    print("-", step)

# The loan program is read out of USER_PROMPT, so it must be visible: it selects which
# column of the document matrix decides each document's R/O relevance.
print("\n--- Loan Program ---")
_detection = result.get("loan_program_detection") or {}
if result.get("loan_program"):
    print(f"{result['loan_program']} "
          f"(matched \"{_detection.get('matched_alias')}\" "
          f"in the {_detection.get('source')})")
elif _detection.get("candidates"):
    print(f"AMBIGUOUS — the request names {', '.join(_detection['candidates'])}. "
          "Using the strongest relevance across all programs.")
    print("Name exactly one program in USER_PROMPT to pin it down.")
else:
    print("Not specified. Using the strongest relevance across all programs "
          "(documents can only be over-prioritised, never dropped).")
    print(f"Name one of {', '.join(_matrix.loan_programs)} in USER_PROMPT to pin it down.")

# A run under a tight provider quota spends most of its time waiting for
# tokens, which is indistinguishable from a hang unless it is reported.
print("\n--- LLM Rate Limit ---")
_rl = result.get("rate_limit") or {}
if _rl:
    print(f"{_rl['llm_calls']} LLM call, giới hạn {_rl['requests_per_minute']}/phút")
    print(f"Không thể nhanh hơn: {_rl['minimum_seconds_for_these_calls']}s")
    print(f"Tổng thời gian bị chặn (cộng dồn mọi luồng): "
          f"{_rl['throttled_seconds_all_threads']}s")
    print("Chỉnh LLM_REQUESTS_PER_MINUTE trong .env nếu quota thay đổi.")

print("\n--- Document Classifications ---")
classification_keys = [
    "filename",
    "document_type",
    "document_group",
    "agent_relevance",
    "loan_program",
    "agent",
    "confidence",
    "reasoning",
    "extraction_status",
    "extraction_error",
    # Which structured-extraction passes ran, and why one did not. The extracted
    # payloads themselves are tens of thousands of characters and go to their own
    # JSON files below — only the status belongs in the printed summary.
    "is_bctc",
    "bctc_extraction_error",
    "is_proposal",
    "proposal_extraction_error",
    "is_cic_s10a",
    "cic_s10a_extraction_error",
    "is_cic_r21",
    "cic_r21_extraction_error",
    "classifier_error_type",
    "classifier_error",
]
for item in result["document_classifications"]:
    classification = {key: item.get(key) for key in classification_keys}
    print(json.dumps(classification, ensure_ascii=False, indent=2))

# A failed extraction drops the agent back to raw OCR: the report still comes
# out, just without the structured figures, so the failure needs to be loud.
_extraction_failures = [
    (item["filename"], label, item.get(f"{prefix}_extraction_error") or "?")
    for item in result["document_classifications"]
    for prefix, flag, label in (
        ("bctc", "is_bctc", "BCTC"),
        ("proposal", "is_proposal", "đề nghị cấp tín dụng"),
        ("cic_s10a", "is_cic_s10a", "CIC S10A"),
        ("cic_r21", "is_cic_r21", "CIC R21"),
    )
    if item.get(flag) and not item.get(f"{prefix}_extraction")
]
if _extraction_failures:
    print(f"\nWARNING: {len(_extraction_failures)} lần trích xuất thất bại — "
          "agent sẽ dùng OCR thô thay cho dữ liệu có cấu trúc:")
    for _name, _label, _why in _extraction_failures:
        print(f"  - {_name} ({_label}): {_why}")

# Documents that matched no matrix row are shared with every agent, which is
# safe but imprecise — each one is a missing keyword or a missing document type.
_unmatched = [
    item["filename"]
    for item in result["document_classifications"]
    if not item.get("document_type")
]
if _unmatched:
    print(
        f"\nWARNING: {len(_unmatched)} document(s) matched no type in the "
        f"matrix and were shared with every agent: {', '.join(_unmatched)}"
    )


In [ ]:
run_id = "_".join([TESTCASE_ID, datetime.now().strftime("%Y%m%d_%H%M%S")])
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

(run_dir / "final_response.md").write_text(
    result["response"],
    encoding="utf-8",
)
(run_dir / "result.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "document_classifications.json").write_text(
    json.dumps(result["document_classifications"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "document_selections.json").write_text(
    json.dumps(result["document_selections"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "financial_metrics.json").write_text(
    json.dumps(result["financial_metrics"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "agent_outputs.json").write_text(
    json.dumps(result["sub_agent_outputs"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "financial_metrics.json").write_text(
    json.dumps(result.get("financial_metrics", {}), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "bctc_extraction.json").write_text(
    json.dumps(
        {
            doc["filename"]: doc.get("bctc_extraction")
            for doc in result["document_classifications"]
            if doc.get("is_bctc")
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

(run_dir / "proposal_extraction.json").write_text(
    json.dumps(
        {
            doc["filename"]: doc.get("proposal_extraction")
            for doc in result["document_classifications"]
            if doc.get("is_proposal")
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

(run_dir / "cic_s10a_extraction.json").write_text(
    json.dumps(
        {
            doc["filename"]: doc.get("cic_s10a_extraction")
            for doc in result["document_classifications"]
            if doc.get("is_cic_s10a")
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

(run_dir / "cic_r21_extraction.json").write_text(
    json.dumps(
        {
            doc["filename"]: doc.get("cic_r21_extraction")
            for doc in result["document_classifications"]
            if doc.get("is_cic_r21")
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

(run_dir / "sitevisit_extraction.json").write_text(
    json.dumps(
        {
            doc["filename"]: doc.get("sitevisit_extraction")
            for doc in result["document_classifications"]
            if doc.get("is_sitevisit")
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# Export the final response (Markdown) to PDF via `markdown` + WeasyPrint.
# On macOS, WeasyPrint's Pango/Cairo bindings live under Homebrew and aren't on
# the default dynamic-linker path, so point DYLD_LIBRARY_PATH at them first.
import platform

if platform.system() == "Darwin":
    for brew_lib in ("/opt/homebrew/lib", "/usr/local/lib"):
        if os.path.isdir(brew_lib):
            os.environ["DYLD_LIBRARY_PATH"] = (
                brew_lib + ":" + os.environ.get("DYLD_LIBRARY_PATH", "")
            )

try:
    import markdown as md_lib
    from weasyprint import HTML

    from src.utils.charts import charts_to_html
    from src.utils.diagrams import mermaid_to_html
    from src.utils.report_style import REPORT_CSS, tag_wide_tables

    # WeasyPrint has no JS, so mermaid blocks must become HTML before render,
    # and linechart blocks must become inline SVG.
    html_body = md_lib.markdown(
        charts_to_html(mermaid_to_html(result["response"])),
        extensions=["tables", "fenced_code", "footnotes"],
        # Number footnotes by where they are REFERENCED, not where they are
        # defined. The default does the latter, which makes a reader meet
        # footnote 3 before footnote 1 whenever the list is ordered any other
        # way. consolidate_footnotes already sorts them; this makes the render
        # correct even if that ever stops being true.
        extension_configs={"footnotes": {"USE_DEFINITION_ORDER": False}},
    )
    # Wide financial tables get a smaller font so they fit the page box.
    html_body = tag_wide_tables(html_body)
    html_doc = (
        '<html><head><meta charset="utf-8"><style>'
        + REPORT_CSS
        + "</style></head><body>"
        + html_body
        + "</body></html>"
    )

    HTML(string=html_doc).write_pdf(str(run_dir / "final_response.pdf"))
    print(f"Saved PDF: {run_dir / 'final_response.pdf'}")
except ImportError:
    print(
        "Skipped PDF export — run `pip install markdown weasyprint` to enable it."
    )

print(f"Saved artifacts to: {run_dir}")